In [1]:
# 1. IMPORT EVERY LIBRARY NEEDED
#----------------------------------------------------#

import requests
import pandas as pd
import geopandas as gpd
import folium
import cmcrameri.cm as cmc
from folium.plugins import MarkerCluster

In [2]:
# 2. ACESS TO THE WEBSITE AND DATA
#----------------------------------------------------#

# Personal Key -> Needed to have access
NASA_API_KEY = "13b5670c616665b924e5aad3a18a24e2"

# Define the API
api_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{NASA_API_KEY}/MODIS_NRT/world/5"

# Send request
response = requests.get(api_url, params=NASA_API_KEY)

# Checking if it works
print(response.status_code) # should receive 200 (means OK), 404 means not working

200


In [3]:
# 3. CLEANING AND FILTERING DATA
#----------------------------------------------------#

# Reading the data
data = pd.read_csv(api_url, sep=",")

#-----------------------------------
# Checking 
#display(data)
#-----------------------------------

# Selecting only Europe and copy it
data_europe = data[(data['longitude'] >= -27) & (data['latitude'] >= 34) & (data['longitude'] <= 43) & (data['latitude'] <= 72)].copy()

#-----------------------------------
# Checking
#display(data_europe)
#-----------------------------------

# Filtering only the attributes needed and copy it
Selec_europe_data = data_europe[["latitude", "longitude", "acq_date", "brightness", "bright_t31", "frp", "daynight", "confidence"]]

#-----------------------------------
# Checking
#display(Selec_europe_data)
#-----------------------------------


# Checking/Looking at the Data  
Selec_europe_data.info()
#Selec_europe_data.sort_values(by = "confidence")

<class 'pandas.core.frame.DataFrame'>
Index: 276 entries, 1465 to 23141
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   latitude    276 non-null    float64
 1   longitude   276 non-null    float64
 2   acq_date    276 non-null    object 
 3   brightness  276 non-null    float64
 4   bright_t31  276 non-null    float64
 5   frp         276 non-null    float64
 6   daynight    276 non-null    object 
 7   confidence  276 non-null    int64  
dtypes: float64(5), int64(1), object(2)
memory usage: 19.4+ KB


In [4]:
# 4. PREPARE THE DATA FOR VISUALISATION 
#----------------------------------------------------#

# Initiating an empty Web App
fire_map = folium.Map(
    location=[55, 20], 
    zoom_start = 4 , # The higher the number, the closer
    tiles= "Stadia.AlidadeSmoothDark") # BaseMap

# Convert the data into a GeoDataFrame (Geographical Coordinates)
wildfire_europe_gdf = gpd.GeoDataFrame(
    Selec_europe_data, # Data i want to convert
    geometry = gpd.points_from_xy(Selec_europe_data["longitude"], Selec_europe_data["latitude"])) #-> x=longitude, y=latitude

#-----------------------------------
# Checking the transformation
#display(wildfire_europe_gdf.head(3)) # -> There should be a new attribute called "geometry"
#-----------------------------------

# Checking and Setting a crs
#print(wildfire_europe_gdf.crs) # ->In this case None
wildfire_europe_gdf = wildfire_europe_gdf.set_crs(epsg=4326) # -> Setting it to a crs 


In [11]:
# 5. VISUALIZE THE DATA - FIRST MAP (COLORED BY FRP)
#----------------------------------------------------#

# Create an empty cluster group -> used later
marker_cluster = MarkerCluster(name="Wildfire Clusters").add_to(fire_map)

# Adding the tooltip (with Cluster)
for idx, row in wildfire_europe_gdf.iterrows():  
    # Extract coordinates
    lat = row.geometry.y
    lon = row.geometry.x

    # Format the tooltip text explicitly
    tooltip_text = (f"Day/Night: {row['daynight']}<b>"
        f"Intensity: {row['frp']}"
        f"Uncertainty: {row['confidence']}"
                   )
    

    # Create the individual marker and add it to the cluster (NOT directly to the map)
    folium.Marker(
        location=[lat, lon],
        icon=folium.Icon(color="red", icon="fire"),
        tooltip=tooltip_text,
    ).add_to(marker_cluster)




# Adding Layer to Map
folium.LayerControl().add_to(fire_map)

# Displaying the map
fire_map 

In [6]:
# VISUALIZE THE DATA - FIRST MAP (COLORED BY CONFIDENCE)
#----------------------------------------------------#

# Create an empty cluster group -> used later
marker_cluster = MarkerCluster(name="Wildfire Clusters").add_to(fire_map)

In [7]:
# VISUALIZE THE DATA - FIRST MAP (COLORED BY DAY OR NIGHT)
#----------------------------------------------------#


# Create an empty cluster group -> used later
marker_cluster = MarkerCluster(name="Wildfire Clusters").add_to(fire_map)